Figure S14: GWAS colocalizations across tissues and traits. The total number of GWAS hits near clusters is the number of unique GWAS lead variants fine-mapped within 1MB of a gene cluster. The number of GWAS hits colocalized per GWAS trait in each tissue is given for eQTLs and for novel pcQTLs (pcQTLs not colocalized by an eQTL).

In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns 
import statsmodels.api as sm
from tqdm.auto import tqdm
tqdm.pandas()

# get outputs from a config file
import yaml
config_path= '/home/klawren/oak/pcqtls/config_old/main_pcqtl.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

import sys
sys.path.append(f'{config["working_dir"]}/{config["code_dir"]}')
from utils import *
from group_signals import load_gwas_coloc

# Set up plotting
plt.rcParams.update({'font.size': 10})
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42

# Load tissue data
tissue_df = load_tissue_df(config)
tissue_ids = load_tissue_ids(config)

Using Python: /home/klawren/.pixi/envs/python/bin/python


In [17]:
gwas_signal_groups = load_across_tissues(config, load_gwas_signal_groups)
gwas_signal_groups['phenotype_id'] = gwas_signal_groups['signal_id'].str.split('-')
gwas_signal_groups_explode = gwas_signal_groups.explode('phenotype_id')

gwas_hits = gwas_signal_groups_explode[gwas_signal_groups_explode['phenotype_id'].str.contains('gwas')]
gwas_hits['gwas_trait'] = gwas_hits['phenotype_id'].str.split('_cluster').str[0].str.strip('gwas_')

/local/scratch/klawren/slrmtmp.49586076/ipykernel_18230/4205081416.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gwas_hits['gwas_trait'] = gwas_hits['phenotype_id'].str.split('_cluster').str[0].str.strip('gwas_')


In [18]:
gwas_coloc = load_gwas_coloc(config)

getting files in directory: /home/klawren/oak/pcqtls/output/proteincoding_main/coloc/gwas_susie_True
found 1482 files
File is empty: /home/klawren/oak/pcqtls/output/proteincoding_main/coloc/gwas_susie_True/Skin_Not_Sun_Exposed_Suprapubic/Skin_Not_Sun_Exposed_Suprapubic.v8.GEFOS_Forearm.gwas_coloc.txt
File is empty: /home/klawren/oak/pcqtls/output/proteincoding_main/coloc/gwas_susie_True/Skin_Not_Sun_Exposed_Suprapubic/Skin_Not_Sun_Exposed_Suprapubic.v8.GPC-NEO-NEUROTICISM.gwas_coloc.txt
File is empty: /home/klawren/oak/pcqtls/output/proteincoding_main/coloc/gwas_susie_True/Skin_Not_Sun_Exposed_Suprapubic/Skin_Not_Sun_Exposed_Suprapubic.v8.EGG_Pubertal_growth_10F.gwas_coloc.txt
File is empty: /home/klawren/oak/pcqtls/output/proteincoding_main/coloc/gwas_susie_True/Skin_Not_Sun_Exposed_Suprapubic/Skin_Not_Sun_Exposed_Suprapubic.v8.PGC_ASD_2017_CEU.gwas_coloc.txt
File is empty: /home/klawren/oak/pcqtls/output/proteincoding_main/coloc/gwas_susie_True/Skin_Not_Sun_Exposed_Suprapubic/Skin_No

In [19]:
num_gwas_hits_tissue = gwas_coloc[['gwas_id', 'hit1', 'tissue_id']].drop_duplicates().groupby(['gwas_id', 'tissue_id']).size().reset_index(name='num_hits')
num_gwas_hits_total = gwas_coloc[['gwas_id', 'hit1']].drop_duplicates().groupby(['gwas_id']).size().reset_index(name='num_hits')


In [20]:
# Calculate total GWAS hits across all tissues for each GWAS, ordered by total count descending
num_gwas_hits_total = gwas_coloc[['gwas_id', 'hit1']].drop_duplicates().groupby(['gwas_id']).size().reset_index(name='num_hits').set_index('gwas_id')
gwas_order = num_gwas_hits_total['num_hits'].sort_values(ascending=False).index

# eQTL-colocalized GWAS hits (per trait, tissue)
trait_tissue_matrix_eqtl = (
    gwas_hits[gwas_hits['num_e_coloc']>0]
    .groupby(['gwas_trait', 'tissue_id']).size()
    .unstack(fill_value=0)
    .reindex(index=gwas_order, fill_value=0)
    .astype(int)
)

# pcQTL-colocalized GWAS hits (per trait, tissue)
trait_tissue_matrix_pcqtl = (
    gwas_hits[gwas_hits['num_e_coloc']==0]
    .groupby(['gwas_trait', 'tissue_id']).size()
    .unstack(fill_value=0)
    .reindex(index=gwas_order, fill_value=0)
    .astype(int)
)

# GWAS hits per GWAS per tissue
gwas_tissue_matrix = (
    gwas_coloc[['gwas_id', 'hit1', 'tissue_id']]
    .drop_duplicates()
    .groupby(['gwas_id', 'tissue_id']).size()
    .unstack(fill_value=0)
    .reindex(index=gwas_order, fill_value=0)
    .astype(int)
)

# Reindex the total hits for sorting
num_gwas_hits_total_ordered = num_gwas_hits_total.reindex(gwas_order).fillna(0).astype(int)

with pd.ExcelWriter(f"{config['working_dir']}/workflow/supplemental_figures/figures/table_s1.xlsx") as writer:
    num_gwas_hits_total_ordered.to_excel(writer, sheet_name="GWAS_hits_total")
    gwas_tissue_matrix.to_excel(writer, sheet_name="GWAS_hits_per_tissue")
    trait_tissue_matrix_eqtl.to_excel(writer, sheet_name="eQTL_coloc_hits")
    trait_tissue_matrix_pcqtl.to_excel(writer, sheet_name="pcQTL_coloc_hits")

print(f"Saved spreadsheet with GWAS hit counts to: {config['working_dir']}/workflow/supplemental_figures/figures/table_s1.xlsx")


Saved spreadsheet with GWAS hit counts to: /home/klawren/oak/pcqtls/workflow/supplemental_figures/figures/table_s1.xlsx
